In [11]:
# ==============================================================================
# Student Knowledge Level Modeling: End-to-End Analysis Pipeline
# Author: Earl Destriza Tavera (Academic Portfolio Submission)
# Data Source: UCI Machine Learning Repository (ID: 257)
# ==============================================================================

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    silhouette_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from ucimlrepo import fetch_ucirepo

# Configure plotting style
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.max_open_warning": 0})

# ==============================================================================
# STEP 1: AUTOMATED DATA INGESTION VIA UCI API
# ==============================================================================
print("--- [STEP 1] Fetching Dataset from UCI Repository (ID: 257) ---")

# Fetch dataset directly from web
dataset = fetch_ucirepo(id=257)

X = dataset.data.features.copy()
y = dataset.data.targets.copy()

# The target variable name in the UCI API return is typically 'UNS'
target_col = y.columns[0]

# Standardize nominal target strings
target_mapping = {
    "very_low": "Very Low",
    "very low": "Very Low",
    "low": "Low",
    "middle": "Middle",
    "high": "High",
}
y_clean = y[target_col].str.strip().str.lower().map(target_mapping)

# Recombine into a unified dataframe for analysis & exporting
full_df = X.copy()
full_df["UNS"] = y_clean

print(
    f"Data successfully fetched from API! Full dataset shape: {full_df.shape}"
)
print("\nTarget Class Distribution:")
print(full_df["UNS"].value_counts())

# Save local backup CSV
full_df.to_csv("uci_user_modeling_full.csv", index=False)
print("Backup copy exported to disk as 'uci_user_modeling_full.csv'\n")


# ==============================================================================
# STEP 2: EXPLORATORY DATA ANALYSIS & VISUALIZATION
# ==============================================================================
print("--- [STEP 2] Generating Exploratory Visualizations ---")

ordered_classes = ["Very Low", "Low", "Middle", "High"]
class_counts = full_df["UNS"].value_counts().reindex(ordered_classes)

# A. Class Distribution Bar Plot
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(
    x=class_counts.index, y=class_counts.values, palette="viridis", ax=ax
)
ax.set_title(
    "Student Knowledge Level (UNS) Class Distribution", fontsize=14, pad=15
)
ax.set_xlabel("Knowledge Mastery Category", fontsize=12)
ax.set_ylabel("Number of Students", fontsize=12)
for p in ax.patches:
    ax.annotate(
        f"{int(p.get_height())}",
        (p.get_x() + p.get_width() / 2.0, p.get_height()),
        ha="center",
        va="bottom",
        fontsize=11,
        xytext=(0, 4),
        textcoords="offset points",
    )
plt.tight_layout()
plt.savefig("class_distribution.png", dpi=300)
plt.close(fig)

# B. Feature Multicollinearity Heatmap
feature_cols = list(X.columns)
corr_matrix = full_df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    cbar_kws={"label": "Pearson Correlation"},
    ax=ax,
)
ax.set_title("Feature Correlation Heatmap", fontsize=14, pad=15)
plt.tight_layout()
plt.savefig("feature_correlations.png", dpi=300)
plt.close(fig)
print(
    "EDA charts saved: 'class_distribution.png', 'feature_correlations.png'.\n"
)


# ==============================================================================
# STEP 3: UNSUPERVISED LEARNING (K-MEANS CLUSTERING)
# ==============================================================================
print("--- [STEP 3] Performing K-Means Clustering Analysis ---")

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

silhouette_scores = {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = km.fit_predict(X_scaled)
    silhouette_scores[k] = silhouette_score(X_scaled, cluster_labels)

best_k = max(silhouette_scores, key=silhouette_scores.get)
print(f"Silhouette Scores across K: {silhouette_scores}")
print(f"Optimal unsupervised cluster separation detected at K={best_k}.")

kmeans4 = KMeans(n_clusters=4, random_state=42, n_init=10)
full_df["Unsupervised_Cluster"] = kmeans4.fit_predict(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.scatterplot(
    data=full_df,
    x="PEG",
    y="LPR",
    hue="UNS",
    hue_order=ordered_classes,
    palette="Set1",
    ax=axes[0],
    s=60,
    alpha=0.85,
)
axes[0].set_title("True Supervised Knowledge Boundaries", fontsize=13)
axes[0].set_xlabel("Goal Exam Score (PEG)")
axes[0].set_ylabel("Related Exam Score (LPR)")

sns.scatterplot(
    data=full_df,
    x="PEG",
    y="LPR",
    hue="Unsupervised_Cluster",
    palette="tab10",
    ax=axes[1],
    s=60,
    alpha=0.85,
)
axes[1].set_title("Unsupervised K-Means Clusters (K=4)", fontsize=13)
axes[1].set_xlabel("Goal Exam Score (PEG)")
axes[1].set_ylabel("Related Exam Score (LPR)")
plt.tight_layout()
plt.savefig("clustering_comparison.png", dpi=300)
plt.close(fig)
print("Clustering comparison plot saved: 'clustering_comparison.png'.\n")


# ==============================================================================
# STEP 4: SUPERVISED LEARNING & PREDICTIVE MODELING
# ==============================================================================
print("--- [STEP 4] Training Supervised Predictive Engines ---")

# Since the API fetches the full unpartitioned dataset (403 rows),
# we apply a stratified

--- [STEP 1] Fetching Dataset from UCI Repository (ID: 257) ---
Data successfully fetched from API! Full dataset shape: (403, 6)

Target Class Distribution:
UNS
Low         129
Middle      122
High        102
Very Low     50
Name: count, dtype: int64
Backup copy exported to disk as 'uci_user_modeling_full.csv'

--- [STEP 2] Generating Exploratory Visualizations ---


/var/folders/ms/mk52mcr16w91dmcsmvl1jkc00000gn/T/ipykernel_87811/3032394591.py:77: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(


EDA charts saved: 'class_distribution.png', 'feature_correlations.png'.

--- [STEP 3] Performing K-Means Clustering Analysis ---
Silhouette Scores across K: {2: 0.18184025762533484, 3: 0.1707724676182014, 4: 0.17240838212507953, 5: 0.1827239603881878, 6: 0.17933194088533685}
Optimal unsupervised cluster separation detected at K=5.
Clustering comparison plot saved: 'clustering_comparison.png'.

--- [STEP 4] Training Supervised Predictive Engines ---
